# 02. Exploratory Data Analysis (EDA) & Visualizations

📌 **Executive Summary**
This notebook explores weekly sales trends, store type characteristics, and holiday performance across Walmart stores from 2010 to 2012. The goal is to extract business insights regarding seasonality, store formatting efficiency, and promotional impacts.

**Key Questions Answered:**
1. **Overall Trend**: How do overall weekly sales fluctuate over time?
2. **Store Formatting**: How do average weekly sales and store sizes differ across Store Types (A, B, C)?
3. **Holiday Impact**: What is the revenue lift associated with holiday weeks compared to non-holiday weeks?

### Step 1: Environment Setup & Cleaned Data Ingestion
Mounting Google Drive and loading the preprocessed `clean_walmart_data.csv` dataset for exploratory analysis.

In [ ]:
# ==========================================
# 1. Mount Google Drive & Import Libraries
# ==========================================
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style and parameters
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/포트폴리오-walmart(캐글)/'

# Load cleaned dataset
df = pd.read_csv(DATA_PATH + 'clean_walmart_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("=== Cleaned Data Loaded Successfully ===")
print(f"Total Records: {len(df):,}")
print(f"Analysis Period: {df['Date'].min().strftime('%Y-%m-%d')} ~ {df['Date'].max().strftime('%Y-%m-%d')}\n")



### Q1. Overall Weekly Sales Trend (2010 - 2012)
Aggregating total weekly sales across all stores to examine macro-level seasonality and identify annual sales peak periods.

In [ ]:
# ==========================================
# 2. Q1. Overall Weekly Sales Trend
# ==========================================
weekly_sales = df.groupby('Date')['Weekly_Sales'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(weekly_sales['Date'], weekly_sales['Weekly_Sales'] / 1e6, color='#0071CE', linewidth=2)
plt.title('Walmart Overall Weekly Sales Trend (2010 - 2012)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=11)
plt.ylabel('Total Weekly Sales ($ Millions)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### Q2. Sales & Store Size Performance by Store Type
Comparing average weekly sales and physical footprint (Sq Ft) across Store Types (A, B, C) to evaluate how store formatting impacts revenue output.

In [ ]:
# ==========================================
# 3. Q2. Sales & Store Size Analysis by Store Type
# ==========================================
type_summary = df.groupby('Type').agg(
    Avg_Weekly_Sales=('Weekly_Sales', 'mean'),
    Total_Sales=('Weekly_Sales', 'sum'),
    Avg_Store_Size=('Size', 'mean'),
    Store_Count=('Store', 'nunique')
).reset_index()

print("--- Summary by Store Type ---")
print(type_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Average sales by Store Type
sns.barplot(data=type_summary, x='Type', y='Avg_Weekly_Sales', palette='Blues_r', ax=axes[0])
axes[0].set_title('Average Weekly Sales by Store Type', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Avg Weekly Sales ($)', fontsize=10)

# Average store size by Store Type
sns.barplot(data=type_summary, x='Type', y='Avg_Store_Size', palette='Greens_r', ax=axes[1])
axes[1].set_title('Average Store Size (Sq Ft) by Store Type', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Size (Sq Ft)', fontsize=10)

plt.tight_layout()
plt.show()



### Q3. Holiday vs. Non-Holiday Revenue Lift
Quantifying the percentage lift in average weekly sales during holiday weeks vs. non-holiday weeks to measure holiday seasonality demand.

In [ ]:
# ==========================================
# 4. Q3. Holiday vs. Non-Holiday Sales Comparison
# ==========================================
holiday_summary = df.groupby('IsHoliday')['Weekly_Sales'].agg(['mean', 'median', 'count']).reset_index()
holiday_summary.columns = ['IsHoliday', 'Mean_Sales', 'Median_Sales', 'Record_Count']

sales_non_holiday = holiday_summary.loc[holiday_summary['IsHoliday'] == False, 'Mean_Sales'].values[0]
sales_holiday = holiday_summary.loc[holiday_summary['IsHoliday'] == True, 'Mean_Sales'].values[0]
pct_diff = ((sales_holiday - sales_non_holiday) / sales_non_holiday) * 100

print("\n--- Sales Comparison: Holiday vs. Non-Holiday ---")
print(f"- Non-Holiday Avg Weekly Sales : ${sales_non_holiday:,.2f}")
print(f"- Holiday Avg Weekly Sales     : ${sales_holiday:,.2f}")
print(f"- Sales Increase on Holidays   : +{pct_diff:.2f}%")